# 2 つの文の論理的な関係を接続表現から推定する (談話関係)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/17_semantics/discourse_relation.ipynb)

NLP クックブック 17-12 ／ 意味解析

2 つの文が「原因と結果」「逆接」「例示」などのどの関係にあるかを推定したい。注釈付きデータを使わず、接続表現の予測として解く。

In [1]:
!pip install -q fugashi unidic-lite datasets

In [2]:
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

name = "tohoku-nlp/bert-base-japanese-v3"
tokenizer = AutoTokenizer.from_pretrained(name)
model = AutoModelForMaskedLM.from_pretrained(name).eval()

# 談話関係ごとの代表的な接続表現
CONNECTIVES = {"原因・理由": ["だから", "そのため", "したがって"], "逆接": ["しかし", "ところが", "だが"], "例示": ["たとえば", "例えば"],
               "付加": ["また", "さらに", "そして"], "言い換え": ["つまり", "すなわち"]}
print({c: tokenizer.tokenize(c) for cs in CONNECTIVES.values() for c in cs})      # 「だから」は「だ / から」のように 2 トークンになるものがある

@torch.no_grad()
def connective_score(first, second, connective):
    """接続表現を入れた文で、その接続表現のトークンを 1 つずつ [MASK] にして予測させ、対数確率の平均をとる (疑似対数尤度)"""
    enc = tokenizer(f"{first}{connective}、{second}", return_tensors="pt")
    ids = enc["input_ids"][0]
    conn_ids = tokenizer(connective, add_special_tokens=False)["input_ids"]
    start = next(i for i in range(len(ids)) if ids[i:i + len(conn_ids)].tolist() == conn_ids)
    batch = ids.repeat(len(conn_ids), 1)
    for k in range(len(conn_ids)):
        batch[k, start + k] = tokenizer.mask_token_id
    logp = model(input_ids=batch).logits.log_softmax(-1)
    return float(sum(logp[k, start + k, conn_ids[k]] for k in range(len(conn_ids)))) / len(conn_ids)     # トークン数で割って、長い表現が不利にならないようにする

def relation_scores(first, second):
    """関係ごとに、その関係の接続表現のうち最も自然なもののスコアをとる"""
    return {rel: max(connective_score(first, second, c) for c in cs) for rel, cs in CONNECTIVES.items()}

def relation(first, second, prior=None):
    """スコアを確率の形に直す。prior を与えると、関係ごとの「どんな文の組でも出やすい度合い」を差し引く"""
    best = relation_scores(first, second)
    if prior:
        best = {rel: v - prior[rel] for rel, v in best.items()}
    z = torch.tensor(list(best.values())).softmax(0)
    return {rel: round(float(p), 3) for rel, p in sorted(zip(best, z), key=lambda x: -x[1])}

examples = [("昨夜は大雪が降った。", "今朝は電車が大幅に遅れた。"), ("彼は毎日遅くまで練習した。", "試合には勝てなかった。"),
            ("この地域には多くの果物がある。", "りんごやぶどうが有名だ。"), ("この店は料理がおいしい。", "値段も手ごろだ。"),
            ("彼は約束の時間に来なかった。", "約束を破ったのだ。"), ("新しい薬の効果が確認された。", "副作用も報告されている。")]
for a, b in examples:
    print(a, "→", b, "\n    ", relation(a, b))

{'だから': ['だ', 'から'], 'そのため': ['その', 'ため'], 'したがって': ['したがっ', 'て'], 'しかし': ['しかし'], 'ところが': ['ところ', 'が'], 'だが': ['だ', 'が'], 'たとえば': ['たとえば'], '例えば': ['例えば'], 'また': ['また'], 'さらに': ['さらに'], 'そして': ['そして'], 'つまり': ['つまり'], 'すなわち': ['すなわち']}
昨夜は大雪が降った。 → 今朝は電車が大幅に遅れた。 
     {'原因・理由': 0.424, '逆接': 0.42, '付加': 0.152, '言い換え': 0.003, '例示': 0.001}
彼は毎日遅くまで練習した。 → 試合には勝てなかった。 
     {'逆接': 0.559, '原因・理由': 0.434, '付加': 0.006, '言い換え': 0.0, '例示': 0.0}
この地域には多くの果物がある。 → りんごやぶどうが有名だ。 
     {'原因・理由': 0.518, '逆接': 0.446, '例示': 0.033, '付加': 0.002, '言い換え': 0.001}
この店は料理がおいしい。 → 値段も手ごろだ。 
     {'原因・理由': 0.348, '逆接': 0.333, '付加': 0.317, '言い換え': 0.002, '例示': 0.0}
彼は約束の時間に来なかった。 → 約束を破ったのだ。 
     {'原因・理由': 0.521, '付加': 0.232, '逆接': 0.143, '言い換え': 0.103, '例示': 0.0}
新しい薬の効果が確認された。 → 副作用も報告されている。 
     {'逆接': 0.68, '原因・理由': 0.275, '付加': 0.045, '言い換え': 0.0, '例示': 0.0}


In [3]:
# 明示的な接続表現のある文の組をコーパスから集めて、正解率を測る (接続表現を隠して当てさせる)
import re
from datasets import load_dataset

ds = load_dataset("globis-university/aozorabunko-clean", split="train")
c2r = {c: rel for rel, cs in CONNECTIVES.items() for c in cs}
pattern = re.compile("([^。「」\n]{12,50}。)(" + "|".join(c2r) + ")、([^。「」\n]{12,50}。)")
data = []
for ex in ds.select(range(3000)):
    if ex["meta"]["文字遣い種別"] == "新字新仮名":
        data += [(m.group(1), m.group(3), c2r[m.group(2)]) for m in pattern.finditer(ex["text"][:20000])]
import random
random.seed(0); random.shuffle(data)
by_rel = {}
for a, b, rel in data:
    by_rel.setdefault(rel, []).append((a, b, rel))
test = [x for rel, items in by_rel.items() for x in items[:60]]          # 関係ごとに最大 60 組
calibration = [x for rel, items in by_rel.items() for x in items[60:72]]  # 補正用に、評価に使わない組を関係ごとに 12 組
print({rel: len(items) for rel, items in by_rel.items()}, "→ 評価に", len(test), "組 / 補正に", len(calibration), "組")

import numpy as np
import pandas as pd

# 補正: 関係ごとのスコアの平均 (= その関係の接続表現が、文の中身と関係なく選ばれやすい度合い) を求める
cal_scores = [relation_scores(a, b) for a, b, _ in calibration]
prior = {rel: float(np.mean([s[rel] for s in cal_scores])) for rel in CONNECTIVES}
print("関係ごとの平均スコア:", {rel: round(v, 2) for rel, v in prior.items()})

gold = [rel for _, _, rel in test]
scores = [relation_scores(a, b) for a, b, _ in test]
for label, p in [("補正なし", None), ("補正あり", prior)]:
    pred = [max(s, key=lambda rel: s[rel] - (p[rel] if p else 0)) for s in scores]
    print(f"{label}: 正解率 {sum(x == g for x, g in zip(pred, gold)) / len(test):.3f}")
print("最も多い関係を常に答えた場合:", round(max(pd.Series(gold).value_counts()) / len(test), 3))
print(pd.crosstab(pd.Series(gold, name="正解"), pd.Series(pred, name="予測 (補正あり)")).to_string())

{'原因・理由': 282, '付加': 3325, '逆接': 1706, '言い換え': 146, '例示': 54} → 評価に 294 組 / 補正に 48 組
関係ごとの平均スコア: {'原因・理由': -0.4, '逆接': -0.84, '例示': -6.83, '付加': -1.92, '言い換え': -4.49}
補正なし: 正解率 0.361
補正あり: 正解率 0.551
最も多い関係を常に答えた場合: 0.204
予測 (補正あり)  付加  例示  原因・理由  言い換え  逆接
正解                                
付加         35  19      1     2   3
例示          2  45      0     6   1
原因・理由       7  24      7    19   3
言い換え        1  20      0    37   2
逆接          4  12      2     4  38


- 談話関係の解析には、接続表現が書かれている場合 (明示的) と、書かれていない場合 (暗黙的) がある。暗黙的な関係の推定は、今も難しい課題。
- 「接続表現を隠して当てる」問題は、テキストから自動的に大量に作れる。これで学習・評価したモデルを暗黙的な関係の推定に転用する方法は、古くから使われている。
- ただし、書き手が接続表現を省略する場面と、明示する場面は性質が違う。明示的な例での成績が、暗黙的な関係でもそのまま出るとは限らない。
- 接続表現は 1 トークンとは限らない (「だから」は「だ / から」)。トークンを 1 つずつ隠して予測させる疑似対数尤度で、長さの違う表現を比べられるようにしている。ただし、表現によってスコアの出やすさが大きく違い、補正しないと特定の関係 (ここでは原因・理由) ばかりが選ばれる。評価に使わない少数の組で関係ごとの平均スコアを求め、差し引いてから比べる。
- 「また」「そして」のような付加の表現は頻度が高く、モデルの予測もそこに偏る。関係ごとに事前確率で割る、関係ごとの再現率を見る、といった補正を検討する。
- 注釈付きのデータとしては、英語の Penn Discourse Treebank、日本語の京都大学ウェブ文書リードコーパスの談話関係注釈がある。精度が必要なら、これらで分類器を学習する。